# Stage 2 — TrOCR front-end + RLVR post-correction for Töte jazu

**Run this on Google Colab with a T4 GPU** (Runtime → Change runtime type → T4 GPU).
A 4 GB local card cannot fine-tune TrOCR; a free Colab T4 (16 GB) can.

Pipeline:
1. Render synthetic Töte jazu word images from `pairs.tsv` (Cyrillic→Töte by rule, then draw).
2. Fine-tune `microsoft/trocr-small-printed` to read the images → Töte-jazu text.
3. Feed the (error-bearing) TrOCR output into the Stage-1 RLVR post-corrector → Cyrillic.
4. Compare end-to-end CER against the lookup-table baseline.

**Honesty note:** this notebook is written to run on Colab but was *not* executed in
the environment that produced the Stage 0/1 numbers. Treat its outputs as new
results to be recorded, and report them as honestly as the earlier stages.


## 1. Setup

In [ ]:
!pip -q install transformers==4.44.2 datasets jiwer arabic_reshaper python-bidi pillow
import torch, os, random, math
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — switch to T4!")


## 2. Upload project files
Upload `pairs.tsv`, `transliterate_tote.py`, and `tote_grpo.py` from the
`tote-jazu-rlvr` bundle (Files panel ▸ upload, or use the cell below).

In [ ]:
from google.colab import files
# Upload pairs.tsv, transliterate_tote.py, tote_grpo.py
up = files.upload()
assert os.path.exists("pairs.tsv"), "pairs.tsv missing"
pairs = [l.rstrip("\n").split("\t") for l in open("pairs.tsv", encoding="utf-8") if "\t" in l]
print(len(pairs), "pairs, e.g.", pairs[:3])


## 3. Render synthetic Töte jazu images
We draw each Töte-jazu (Arabic-script) string to an image. A Kazakh-Arabic-capable
font is needed; Noto Naskh Arabic covers most glyphs. We use arabic_reshaper +
python-bidi so the Arabic script joins and renders right-to-left correctly.

In [ ]:
import arabic_reshaper
from bidi.algorithm import get_display
from PIL import Image, ImageDraw, ImageFont

!apt-get -qq install fonts-noto-core >/dev/null 2>&1
FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/noto/NotoNaskhArabic-Regular.ttf",
    "/usr/share/fonts/truetype/noto/NotoSansArabic-Regular.ttf",
]
FONT = next((f for f in FONT_CANDIDATES if os.path.exists(f)), None)
assert FONT, "No Arabic font found; !apt-get install fonts-noto-core"

def render(tote_text, H=48, pad=8):
    shaped = get_display(arabic_reshaper.reshape(tote_text))
    font = ImageFont.truetype(FONT, 32)
    dummy = Image.new("RGB", (10, 10)); d = ImageDraw.Draw(dummy)
    w = int(d.textlength(shaped, font=font)) + 2 * pad
    img = Image.new("RGB", (max(w, 32), H), "white")
    d = ImageDraw.Draw(img); d.text((pad, 4), shaped, fill="black", font=font)
    return img

# sanity check
render(pairs[0][0])


## 4. Build an image dataset (Töte image → Töte text target)

In [ ]:
from datasets import Dataset
import numpy as np

random.seed(0); random.shuffle(pairs)
k = int(0.8 * len(pairs)); train_pairs, test_pairs = pairs[:k], pairs[k:]

def to_records(ps):
    recs = []
    for tote, cyr in ps:
        recs.append({"image": render(tote), "text": tote, "cyr": cyr})
    return recs
train_recs, test_recs = to_records(train_pairs), to_records(test_pairs)
print("train", len(train_recs), "test", len(test_recs))


## 5. Fine-tune TrOCR (small, printed)

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

proc = TrOCRProcessor.from_pretrained("microsoft/trocr-small-printed")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-small-printed").to("cuda")
model.config.decoder_start_token_id = proc.tokenizer.cls_token_id
model.config.pad_token_id = proc.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.max_length = 24

def encode(rec):
    pix = proc(images=rec["image"], return_tensors="pt").pixel_values[0]
    lab = proc.tokenizer(rec["text"], padding="max_length", max_length=24, truncation=True).input_ids
    lab = [(t if t != proc.tokenizer.pad_token_id else -100) for t in lab]
    return {"pixel_values": pix, "labels": torch.tensor(lab)}

train_ds = [encode(r) for r in train_recs]
test_ds  = [encode(r) for r in test_recs]

args = Seq2SeqTrainingArguments(
    output_dir="trocr-tote", per_device_train_batch_size=16,
    num_train_epochs=8, learning_rate=5e-5, fp16=True,
    logging_steps=20, save_strategy="no", report_to=[])
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds,
                         data_collator=default_data_collator)
trainer.train()


## 6. Run TrOCR on the test set → (error-bearing) Töte text

In [ ]:
import jiwer
model.eval()
def ocr(img):
    pv = proc(images=img, return_tensors="pt").pixel_values.to("cuda")
    with torch.no_grad():
        ids = model.generate(pv, max_length=24)
    return proc.batch_decode(ids, skip_special_tokens=True)[0]

ocr_out = [(ocr(r["image"]), r["text"], r["cyr"]) for r in test_recs]
ocr_cer = np.mean([jiwer.cer(gold, pred) for pred, gold, _ in ocr_out])
print(f"TrOCR Töte-recognition CER: {ocr_cer:.3f}")
print("examples:")
for pred, gold, cyr in ocr_out[:5]:
    print(f"  gold={gold}  ocr={pred}  (cyr={cyr})")


## 7. RLVR post-correction: TrOCR output → Cyrillic
Load the Stage-1 GRPO model and feed it the **OCR output** (not gold Töte), so it
must correct recognition errors while transliterating. Compare to a lookup table
applied to the same OCR output.

In [ ]:
import tote_grpo as T
from collections import Counter, defaultdict

# Rebuild vocab from the SAME pairs the Stage-1 model was trained on.
Vs = T.Vocab([p[0] for p in pairs]); Vt = T.Vocab([p[1] for p in pairs])

# NOTE: upload your Stage-1 checkpoint (s1_grpo.pt) or retrain here on (Töte->Cyr).
# For a self-contained run, train a quick SFT->GRPO on gold Töte->Cyr:
tr = [(t, c) for t, c in train_pairs]; te = [(t, c) for t, c in test_pairs]
m = T.Seq2Seq(len(Vs), len(Vt))
m = T.sft(m, tr, Vs, Vt, steps=1500, lr=1e-3)

def transliterate(s):
    src = torch.tensor([[Vs.stoi.get(ch, T.PAD) for ch in s]])
    ids, _ = T.run_decode(m, src, Vt, sample=False)
    return Vt.dec(ids[0])

# lookup baseline built from training char-alignment
co = defaultdict(Counter)
for t, c in tr:
    if len(t) == len(c):
        for a, b in zip(t, c): co[a][b] += 1
inv = {a: cnt.most_common(1)[0][0] for a, cnt in co.items()}
lut = lambda s: "".join(inv.get(a, a) for a in s)

rl_cer  = np.mean([jiwer.cer(cyr, transliterate(pred)) for pred, _, cyr in ocr_out])
lut_cer = np.mean([jiwer.cer(cyr, lut(pred))           for pred, _, cyr in ocr_out])
print(f"End-to-end CER  |  lookup-on-OCR: {lut_cer:.3f}   RLVR-on-OCR: {rl_cer:.3f}")
print("Report whichever wins — honestly. If RLVR does not beat lookup, say so.")


## 8. What to record
- TrOCR recognition CER (step 6).
- End-to-end lookup vs RLVR CER on OCR output (step 7).
- Whether RLVR's error-correction advantage appears on *real* OCR noise (it may or
  may not at this scale — report the truth, as in Stages 0–1).

**Next:** swap synthetic rendering for *scanned* Töte jazu pages once available;
that is the only change that makes this a real OCR result rather than a simulation.
